# Transformer alone (config TXM, with TXD and TXP): cross-validation 1981-2013

Kaggle notebook (GPU T4, dataset `divyanshyecho/profa-inputs-ext` = the files in `src/` plus the model inputs).
The cell output below is the printed output of the Kaggle run, restored from its run log (`kaggle kernels output` keeps the log, not the executed notebook).


In [1]:
import os, glob, subprocess, sys, time, json
import numpy as np, pandas as pd
ROUND, CFGS, SEEDS, KIND, EPOCHS = "TX2", "TXD,TXP,TXM".split(","), 3, "cv", ""
CHAINS, FIXCHAIN = [c for c in "".split(",") if c], "fc"
find = lambda n: sorted(glob.glob(f"/kaggle/input/**/{n}", recursive=True))[0]
BASE = os.path.dirname(find("profA.py")); OUT = "/kaggle/working/out"; os.makedirs(OUT, exist_ok=True)
ENV = dict(os.environ, MA_DATA="/kaggle/input", PROFA_IN=os.path.dirname(find("maps.npz")), PROFA_OLR=find("olr_daily_19*_merged.nc"),
           MODELA_SRC=find("kaggle_modelA.py"), PROFA_OUT=OUT, THREADS="2", PYTHONUNBUFFERED="1")
ENV.update(json.loads(r"""{"CHAIN_PEN":"1","RAINLIN":"10","SYNLIN":"10","RAIN_LEADS":"10","CTRL_SEEDS":"0","FINAL_SNAPS":"1","SEED_BASE":"2000","NO_VO":"1","MIN_ISS_YEAR":"1960","WK_L2":"[1000, 1000]","WK_LAM":"[0.3, 3]"}"""))                                 # extra env for this round (e.g. CHAIN_PEN)
if glob.glob("/kaggle/input/**/ecmwf_rain.npz", recursive=True):
    ENV["PROFB_EC"] = find("ecmwf_rain.npz")                                # Prof B: ECMWF rain reforecast features
if glob.glob("/kaggle/input/**/s2s_feats.npz", recursive=True):
    ENV["PROFB_WIND"] = find("s2s_feats.npz")                               # Prof B1: ECMWF wind PCs
t0 = time.time()
def run(extra, tag):
    e = dict(ENV, **extra); f = open(os.path.join(OUT, f"log_{tag}.txt"), "w")
    return subprocess.Popen([sys.executable, os.path.join(BASE, "profA.py")], env=e, stdout=f, stderr=subprocess.STDOUT)
def wait_all(ps):
    for p in ps: p.wait()
# 1. linear link: l2 grid on the 5 folds -> choose l2 (logistic, by RPS) and alpha (BSISO ridge, by PC MSE)
if not os.path.exists(os.path.join(OUT, "lin_cv.pkl")):
    wait_all([run({"MODE": "lin"}, "lin")])
print(open(os.path.join(OUT, "log_lin.txt")).read()[-1500:])
R = pd.DataFrame(pd.read_pickle(os.path.join(OUT, "lin_cv.pkl")))
grid = sorted(R.l2.unique())
rps_l = np.stack([np.stack(R[R.l2 == l].rps).mean(0) for l in grid])        # (grid, lead): out-of-fold RPS per lead
mse_l = np.stack([np.stack(R[R.l2 == l].pc_mse).mean(0) for l in grid])     # (grid, lead): out-of-fold BSISO MSE per lead
L2 = [float(grid[i]) for i in rps_l.argmin(0)]; ALPHA = [float(grid[i]) for i in mse_l.argmin(0)]
print("chosen l2 per lead:", L2, flush=True); print("chosen BSISO alpha per lead:", ALPHA, flush=True)
json.dump({"L2": L2, "ALPHA": ALPHA}, open(os.path.join(OUT, "linear_choice.json"), "w"), indent=1)
CHAIN = FIXCHAIN or "none"
if CHAINS:                                   # BSISO chain variants of the linear link; the ridge alpha stays as chosen above
    def lin_choice(R):
        g_ = sorted(R.l2.unique()); rp = np.stack([np.stack(R[R.l2 == l].rps).mean(0) for l in g_]); rc = np.stack(R[R.l2 == g_[0]].rps_clim).mean(0)
        return [float(g_[i]) for i in rp.argmin(0)], float(1 - rp.min(0).mean() / rc.mean())
    ps = [run({"MODE": "lin", "CHAIN": c, "ALPHA_CHAIN": json.dumps(ALPHA)}, f"lin_{c}") for c in CHAINS if not os.path.exists(os.path.join(OUT, f"lin_cv_{c}.pkl"))]
    wait_all(ps)
    cand = {"none": lin_choice(R)}
    for c in CHAINS:
        print(open(os.path.join(OUT, f"log_lin_{c}.txt")).read()[-900:])
        cand[c] = lin_choice(pd.DataFrame(pd.read_pickle(os.path.join(OUT, f"lin_cv_{c}.pkl"))))
    for c, (l2c, sk) in cand.items():
        print(f"linear link, chain {c}: out-of-fold RPSS days 1-14 {sk:.4f} | l2 {l2c}", flush=True)
    CHAIN = max(cand, key=lambda c: cand[c][1]); L2 = cand[CHAIN][0]
    print("CHOSEN chain variant:", CHAIN, flush=True)
    json.dump({"L2": L2, "ALPHA": ALPHA, "CHAIN": CHAIN, "cand": cand}, open(os.path.join(OUT, "linear_choice.json"), "w"), indent=1)
elif FIXCHAIN:
    lc = json.load(open(find("linear_choice_fixed.json"))); L2 = lc["L2"]; assert lc["CHAIN"] == FIXCHAIN
L2, ALPHA = json.dumps(L2), json.dumps(ALPHA)
if KIND == "cv":
    SEEDSTR = ",".join(str(s) for s in range(SEEDS))                    # one job per (config, fold): seeds share the fold build
    jobs = [({"MODE": "cv", "CFG": c, "FOLD": str(f), "SEED": SEEDSTR, "L2": L2, "ALPHA": ALPHA, "CHAIN": CHAIN}, f"{c}_f{f}")
            for c in CFGS for f in range(5)]
    running = []
    for extra, tag in jobs:
        while len(running) >= 3:
            running = [p for p in running if p.poll() is None]; time.sleep(5)
        running.append(run(extra, tag)); print(f"started {tag} ({(time.time() - t0) / 60:.1f} min)", flush=True)
    wait_all(running)
    for extra, tag in jobs:
        print(tag, "|", [l for l in open(os.path.join(OUT, f"log_{tag}.txt")).read().strip().splitlines() if l.startswith("CV ") or "Error" in l][-4:], flush=True)
    r = subprocess.run([sys.executable, os.path.join(BASE, "profA_eval.py"), "cv"], env=ENV, capture_output=True, text=True)
    print(r.stdout[-8000:], r.stderr[-3000:])
else:
    EPL = EPOCHS.split(",") if "," in EPOCHS else [EPOCHS] * len(CFGS)   # one stop epoch per config (ensembles)
    ps = []
    for c, ep_ in zip(CFGS, EPL):
        while len([p for p in ps if p.poll() is None]) >= 3:
            time.sleep(5)
        ps.append(run({"MODE": "final", "CFG": c, "SEEDS": str(SEEDS), "EPOCHS": ep_, "L2": L2, "ALPHA": ALPHA, "CHAIN": CHAIN}, f"final_{c}"))
    wait_all(ps)
    for c in CFGS:
        print(open(os.path.join(OUT, f"log_final_{c}.txt")).read()[-1500:])
print(f"round {ROUND} done in {(time.time() - t0) / 60:.1f} min")

device cuda | days 12078 | test years 1990-2007 | seeds 3
samples 7085 | X_BS (7085, 60, 4) X_RN (7085, 10, 26) X_SL (7085, 5)
fields: {'olr': (12078, 17, 49), 'u850': (12078, 7, 15), 'u200': (12078, 7, 15), 'mslp': (12078, 14, 17)}
fold 0 done (0.3 min)
fold 1 done (0.5 min)
fold 2 done (0.8 min)
fold 3 done (1.0 min)
fold 4 done (1.2 min)
l2      10: RPSS days 1-7 0.0924  days 8-14 0.0140  PC MSE d8-14 0.943
l2      30: RPSS days 1-7 0.0982  days 8-14 0.0196  PC MSE d8-14 0.942
l2     100: RPSS days 1-7 0.1023  days 8-14 0.0248  PC MSE d8-14 0.942
l2     300: RPSS days 1-7 0.1014  days 8-14 0.0269  PC MSE d8-14 0.940
l2    1000: RPSS days 1-7 0.0915  days 8-14 0.0236  PC MSE d8-14 0.937
l2    3000: RPSS days 1-7 0.0707  days 8-14 0.0150  PC MSE d8-14 0.935
total 1.2 min

chosen l2 per lead: [10.0, 100.0, 300.0, 300.0, 300.0, 300.0, 300.0, 300.0, 300.0, 300.0, 300.0, 300.0, 300.0, 300.0]
chosen BSISO alpha per lead: [10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 1000.0, 1000.0, 3000.0, 30

/usr/local/lib/python3.13/dist-packages/mistune.py:435: SyntaxWarning: invalid escape sequence '\|'
  cells[i][c] = re.sub('\\\\\|', '|', cell)
